In [8]:
import pandas as pd
import time
from PIL import Image
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import os
import math

# Constante para el radio de la Tierra en kilómetros.
R_TIERRA_KM = 6371
# Define la carpeta de destino para todas las imágenes
CARPETA_DESTINO = r"C:\imagenes EPH"

def take_image10K_manual(lat, lon, filename):
    """
    Toma una captura de pantalla de Google Earth y la guarda con el nombre de archivo dado.
    """
    options = webdriver.ChromeOptions()
    options.add_argument("--window-size=2920,3220")
    options.add_argument("--headless")

    driver = webdriver.Chrome(options=options)

    try:
        url = f"https://earth.google.com/web/@{lat},{lon},20a,300000d,1y,-0h,0t,0r/data=CgRCAggBOgMKATBCAggASg0I____________ARAA"
        driver.get(url)
        
        # Espera para que la página cargue completamente
        time.sleep(100)
        
        # Guarda la captura de pantalla
        driver.save_screenshot(filename)
        
        # Recorte de la imagen
        img = Image.open(filename)
        width, height = img.size
        crop_top = 200
        crop_bottom = 100
        cropped_img = img.crop((0, crop_top, width, height - crop_bottom))
        cropped_img.save(filename)

        print(f"✅ Captura de pantalla guardada en: {filename}")

    except Exception as e:
        print(f"❌ Error al guardar la imagen: {filename}. Error: {e}")

    finally:
        driver.quit()

def mover_coordenadas(lat, lon, distancia_km, angulo_grados):
    """
    Calcula nuevas coordenadas (lat, lon) a una distancia y ángulo dados.
    """
    lat_rad = math.radians(lat)
    lon_rad = math.radians(lon)
    angulo_rad = math.radians(angulo_grados)

    nueva_lat_rad = math.asin(math.sin(lat_rad) * math.cos(distancia_km / R_TIERRA_KM) +
                              math.cos(lat_rad) * math.sin(distancia_km / R_TIERRA_KM) * math.cos(angulo_rad))
    nueva_lon_rad = lon_rad + math.atan2(math.sin(angulo_rad) * math.sin(distancia_km / R_TIERRA_KM) * math.cos(lat_rad),
                                        math.cos(distancia_km / R_TIERRA_KM) - math.sin(lat_rad) * math.sin(nueva_lat_rad))

    return math.degrees(nueva_lat_rad), math.degrees(nueva_lon_rad)

def generar_capturas_desplazadas(lat_centro, lon_centro, nombre_base, desplazamiento_km, escala):
    """
    Genera 8 capturas de pantalla alrededor de un punto central
    y solo baja las que faltan.
    """
    desplazamientos = {
        "NorteOeste": {"distancia": math.sqrt(desplazamiento_km**2 + desplazamiento_km**2), "angulo": 315},
        "Norte": {"distancia": desplazamiento_km, "angulo": 0},
        "Centro": {"distancia": 0, "angulo": 0},
        "NorteEste": {"distancia": math.sqrt(desplazamiento_km**2 + desplazamiento_km**2), "angulo": 45},
        "Oeste": {"distancia": desplazamiento_km, "angulo": 270},
        "Este": {"distancia": desplazamiento_km, "angulo": 90},
        "SurOeste": {"distancia": math.sqrt(desplazamiento_km**2 + desplazamiento_km**2), "angulo": 225},
        "Sur": {"distancia": desplazamiento_km, "angulo": 180},
        "SurEste": {"distancia": math.sqrt(desplazamiento_km**2 + desplazamiento_km**2), "angulo": 135},
    }
    
    # Crea la carpeta de destino principal si no existe
    os.makedirs(CARPETA_DESTINO, exist_ok=True)

    # Iterar sobre cada desplazamiento
    for direccion, params in desplazamientos.items():
        distancia = params["distancia"]
        angulo = params["angulo"]
        
        nueva_lat, nueva_lon = mover_coordenadas(lat_centro, lon_centro, distancia, angulo)
        
        # Construye el nombre de archivo completo
        nombre_archivo = f"{nombre_base}_{escala}_{direccion}.png".replace(":", "_").replace("/", ".")
        ruta_completa = os.path.join(CARPETA_DESTINO, nombre_archivo)
        
        # ---------------------------------------------
        # Lógica de verificación de existencia del archivo
        # ---------------------------------------------
        if os.path.exists(ruta_completa):
            print(f"⏩ La imagen ya existe, saltando: {nombre_archivo}")
        else:
            print(f"\nGenerando imagen para {nombre_base} ({escala}) - {direccion}...")
            take_image10K_manual(nueva_lat, nueva_lon, ruta_completa)

# ---
# Lógica principal
# ---

# Cargar el DataFrame
try:
    ciudades = pd.read_csv('J:\EPH sensibilidad\EPH con latlon.csv', encoding='latin1', sep=';')
    print("✅ Archivo CSV cargado correctamente.")
except Exception as e:
    print(f"❌ Error al cargar el archivo CSV: {e}")
    exit()

# Recorrer cada fila del DataFrame y generar las capturas
for index, row in ciudades.iterrows():
    ciudad_nombre = row['City']
    
    try:
        lat_str = str(row['Latitude']).replace(',', '.')
        lon_str = str(row['Longitude']).replace(',', '.')
        
        lat = float(lat_str)
        lon = float(lon_str)
        
    except ValueError:
        print(f"❌ Error: La latitud o longitud para '{ciudad_nombre}' no son números válidos.")
        continue
    
    # Formatear el nombre de la ciudad para evitar espacios en los nombres de archivo
    nombre_ciudad_formateado = ciudad_nombre.replace(" ", "_").replace(".", "")
    
    print(f"\n--- Procesando ciudad: {ciudad_nombre} ---")
    
    # Llama a la función para 10KM
    print("Iniciando descargas para el set de 10K...")
    generar_capturas_desplazadas(lat, lon, nombre_ciudad_formateado, 1, "10K")
    
    # Llama a la función para 5KM
    print("\nIniciando descargas para el set de 5K...")
    generar_capturas_desplazadas(lat, lon, nombre_ciudad_formateado, 0.5, "5K")

print("\n🚀 ¡Proceso de descarga de imágenes completado!")

✅ Archivo CSV cargado correctamente.

--- Procesando ciudad: Gran La Plata ---
Iniciando descargas para el set de 10K...
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_NorteOeste.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_Norte.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_Centro.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_NorteEste.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_Oeste.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_Este.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_SurOeste.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_Sur.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_10K_SurEste.png

Iniciando descargas para el set de 5K...
⏩ La imagen ya existe, saltando: Gran_La_Plata_5K_NorteOeste.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_5K_Norte.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_5K_Centro.png
⏩ La imagen ya existe, saltando: Gran_La_Plata_5K_NorteEste.png
⏩ La imagen ya existe, sal

⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_10K_NorteEste.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_10K_Oeste.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_10K_Este.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_10K_SurOeste.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_10K_Sur.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_10K_SurEste.png

Iniciando descargas para el set de 5K...
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_5K_NorteOeste.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_5K_Norte.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_5K_Centro.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_5K_NorteEste.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_5K_Oeste.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_5K_Este.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_5K_SurOeste.png
⏩ La imagen ya existe, saltando: Ushuaia-Río_Grande_5K_Sur.png
⏩ La imagen ya existe, sal

⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_NorteOeste.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_Norte.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_Centro.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_NorteEste.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_Oeste.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_Este.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_SurOeste.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_Sur.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_10K_SurEste.png

Iniciando descargas para el set de 5K...
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_5K_NorteOeste.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_5K_Norte.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_5K_Centro.png
⏩ La imagen ya existe, saltando: Gran_Tucumán-Tafí_Viejo_5K_NorteEste.png
⏩ La imagen ya ex

⏩ La imagen ya existe, saltando: Corrientes_10K_NorteEste.png
⏩ La imagen ya existe, saltando: Corrientes_10K_Oeste.png
⏩ La imagen ya existe, saltando: Corrientes_10K_Este.png
⏩ La imagen ya existe, saltando: Corrientes_10K_SurOeste.png
⏩ La imagen ya existe, saltando: Corrientes_10K_Sur.png
⏩ La imagen ya existe, saltando: Corrientes_10K_SurEste.png

Iniciando descargas para el set de 5K...
⏩ La imagen ya existe, saltando: Corrientes_5K_NorteOeste.png
⏩ La imagen ya existe, saltando: Corrientes_5K_Norte.png
⏩ La imagen ya existe, saltando: Corrientes_5K_Centro.png
⏩ La imagen ya existe, saltando: Corrientes_5K_NorteEste.png
⏩ La imagen ya existe, saltando: Corrientes_5K_Oeste.png
⏩ La imagen ya existe, saltando: Corrientes_5K_Este.png
⏩ La imagen ya existe, saltando: Corrientes_5K_SurOeste.png
⏩ La imagen ya existe, saltando: Corrientes_5K_Sur.png
⏩ La imagen ya existe, saltando: Corrientes_5K_SurEste.png

--- Procesando ciudad: Gran San Luis ---
Iniciando descargas para el set de 10